# Covariance Matrix

Let $R_1, R_2, ..., R_k$ be $k$ random variables. The _covariance_ between $R_i$ and $R_j$ is given by:

$$ \text{Cov}(R_i, R_j) = \mathbf{E}[(R_i - \mathbf{E}[R_i])(R_j - \mathbf{E}[R_j])] $$

where $\mathbf{E}[X]$ denotes the expectation of the random variable $X$.

Suppose we have data of size $N$, i.e., for each random variable $R_i$, we have a sample of size $N$. Then,

$$ \text{Cov}(R_i, R_j) = \frac{1}{N - 1} \sum_{d = 1}^{N} (R_{i, d} - \mathbf{E}[R_i])(R_{j, d} - \mathbf{E}[R_j])$$

where $R_{i, d}$ denotes the $d$-th observation of $R_i$. 

**Goal**: Given a $k \times N$ matrix $A$, where $A_{i, d}$ denotes the $d$-th observation of the random variable $R_i$, we want to calculate the $k \times k$ covariance matrix $\Sigma$, where $\Sigma_{i, j} = \text{Cov}(R_i, R_j)$.

## 1. Python

The following function takes as input a matrix $A$ and returns the covariance matrix. It mostly avoids using NumPy vectorization. 

In [1]:
import numpy as np

In [2]:
'''
Calculating the covariance matrix of a given matrix A.
A: ndarray
'''

def cov_py(A):
        
    k, N = A.shape
    C = np.zeros((k, k))

    mean = np.zeros((k)) # a 1D array to store the mean of every row
    
    for row in range(k):
        sum_row = 0                  
        
        for j in range(N):
            sum_row += A[row, j]
        
        mean[row] = sum_row / N

    for i in range(k):
        for j in range(k):
            sum_p = 0
            
            for d in range(N):
                sum_p += (A[i, d] - mean[i]) * (A[j, d] - mean[j])
            
            C[i, j] = sum_p / (N - 1)

    return C    

Consider the following example of a $50 \times 800$ matrix:

In [3]:
k = 50
N = 800

A = np.random.rand(k, N) * 10  # creates a k x N 2D array of random values

np.set_printoptions(precision=3, suppress=True)  # 3 decimal places, suppress=True to suppress scientific notation

In [4]:
cov_py(A)

array([[ 8.042, -0.1  , -0.156, ..., -0.057,  0.106, -0.009],
       [-0.1  ,  8.518, -0.408, ..., -0.16 ,  0.012, -0.441],
       [-0.156, -0.408,  8.1  , ..., -0.066,  0.   ,  0.724],
       ...,
       [-0.057, -0.16 , -0.066, ...,  8.358,  0.248, -0.763],
       [ 0.106,  0.012,  0.   , ...,  0.248,  8.04 , -0.479],
       [-0.009, -0.441,  0.724, ..., -0.763, -0.479,  7.971]],
      shape=(50, 50))

**Performance:**

In [5]:
%timeit cov_py(A)

1.38 s ± 81.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## 2. Numba

We can improve the performance of the function `cov_py` using Numba as follows:

In [6]:
import numba as nb

In [7]:
cov_py_nb = nb.njit(cov_py)

In [8]:
cov_py_nb(A)

array([[ 8.042, -0.1  , -0.156, ..., -0.057,  0.106, -0.009],
       [-0.1  ,  8.518, -0.408, ..., -0.16 ,  0.012, -0.441],
       [-0.156, -0.408,  8.1  , ..., -0.066,  0.   ,  0.724],
       ...,
       [-0.057, -0.16 , -0.066, ...,  8.358,  0.248, -0.763],
       [ 0.106,  0.012,  0.   , ...,  0.248,  8.04 , -0.479],
       [-0.009, -0.441,  0.724, ..., -0.763, -0.479,  7.971]],
      shape=(50, 50))

**Performance:**

In [9]:
%timeit cov_py_nb(A)

1.98 ms ± 177 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Hence, the Numba version is faster by orders of magnitude.

## 3. NumPy

As we saw earlier, 

$$ \text{Cov}(R_i, R_j) = \frac{1}{N - 1} \sum_{d = 1}^{N} (R_{i, d} - \mathbf{E}[R_i])(R_{j, d} - \mathbf{E}[R_j]).$$

Thus, we can first calculate the _centered_ matrix: we shift each row left by its mean. That is, in the new matrix $A_c$, we have $R'_{i, d} = R_{i, d} - \mathbf{E}[R_i]$. Then, $\sum_{d = 1}^{N} (R_{i, d} - \mathbf{E}[R_i])(R_{j, d} - \mathbf{E}[R_j])$ is given by $A_c  A_c^\top$. We use this in the following function to vectorize the calculations needed to obtain the covariance matrix:

In [10]:
'''
Calculating the covariance matrix of a given matrix A.
A: ndarray
'''

def cov_np(A):
    k, N = A.shape
    C = np.zeros((k, k))

    mean = np.mean(A, axis=1)  # stores the mean of each row of A (axis=1 takes the mean along the rows)
    
    A_c = A - mean[:, None]  # centered matrix
    C = (A_c @ A_c.T) / (N - 1)  

    return C

In [11]:
cov_np(A)

array([[ 8.042, -0.1  , -0.156, ..., -0.057,  0.106, -0.009],
       [-0.1  ,  8.518, -0.408, ..., -0.16 ,  0.012, -0.441],
       [-0.156, -0.408,  8.1  , ..., -0.066,  0.   ,  0.724],
       ...,
       [-0.057, -0.16 , -0.066, ...,  8.358,  0.248, -0.763],
       [ 0.106,  0.012,  0.   , ...,  0.248,  8.04 , -0.479],
       [-0.009, -0.441,  0.724, ..., -0.763, -0.479,  7.971]],
      shape=(50, 50))

**Performance:**

In [12]:
%timeit cov_np(A)

233 μs ± 109 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 4. Using `np.cov`

`np.cov` is a highly optimized function in the NumPy library that returns the covariance matrix of the given matrix:

In [13]:
np.cov(A)

array([[ 8.042, -0.1  , -0.156, ..., -0.057,  0.106, -0.009],
       [-0.1  ,  8.518, -0.408, ..., -0.16 ,  0.012, -0.441],
       [-0.156, -0.408,  8.1  , ..., -0.066,  0.   ,  0.724],
       ...,
       [-0.057, -0.16 , -0.066, ...,  8.358,  0.248, -0.763],
       [ 0.106,  0.012,  0.   , ...,  0.248,  8.04 , -0.479],
       [-0.009, -0.441,  0.724, ..., -0.763, -0.479,  7.971]],
      shape=(50, 50))

In [14]:
%timeit np.cov(A)

382 μs ± 107 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
